SETUP AND INPORTS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.helpers import *            # load_csv_data, create_csv_submission
from src.implementations import *    # your 6 functions from Table 1

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LOAD THE DATA

In [11]:
DATA_PATH = "data/"   # folder containing x_train.csv, y_train.csv, x_test.csv
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print(x_train.shape, x_test.shape, y_train.shape)
print("Fraction of positive labels:", np.mean(y_train == 1))

(328135, 321) (109379, 321) (328135,)
Fraction of positive labels: 0.08830207079403295


In [26]:
import pandas as pd

df_x_train = pd.read_csv("data/x_train.csv")
df_y_train = pd.read_csv("data/y_train.csv")
df_x_test = pd.read_csv("data/x_test.csv")



In [25]:
df_x_train.head()

,Id,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,...,_PAREC1,_PASTAE1,_LMTACT1,_LMTWRK1,_LMTSCL1,_RFSEAT2,_RFSEAT3,_FLSHOT6,_PNEUMO2,_AIDTST3
0,0,53,11,11162015,11,16,2015,1100,2015015629,2015015629,...,1,1,3.0,3.0,4.0,1,1,NaN,NaN,2.0
1,1,33,12,12152015,12,15,2015,1200,2015004387,2015004387,...,9,9,3.0,3.0,4.0,9,9,NaN,NaN,NaN
2,2,20,10,10202015,10,20,2015,1100,2015005638,2015005638,...,4,2,2.0,2.0,3.0,1,1,1.0,2.0,2.0
3,3,42,6,6182015,6,18,2015,1100,2015004694,2015004694,...,2,2,2.0,2.0,3.0,1,1,2.0,2.0,2.0
4,4,24,11,11062015,11,6,2015,1100,2015004673,2015004673,...,9,9,3.0,3.0,4.0,1,1,9.0,9.0,2.0


MINIMAL PREPOCESSING

In [12]:
# Replace missing values (NaN) by the column median computed on the TRAIN set
median = np.nan_to_num(np.nanmedian(x_train, axis=0))
x_train = np.where(np.isnan(x_train), median, x_train)
x_test = np.where(np.isnan(x_test), median, x_test)

# Standardize with train statistics
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
std[std == 0] = 1                     # avoid division by zero
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

# Add the offset (bias) column -> tx, as in the labs
tx_train = np.c_[np.ones(x_train.shape[0]), x_train]
tx_test = np.c_[np.ones(x_test.shape[0]), x_test]

# Labels in {0,1} for logistic regression
y = (y_train + 1) / 2

LOCAL TRAIN/VALIDATION SPLIT

In [13]:
def split_data(y, tx, ratio, seed=1):
    """Randomly split the data into train and validation sets."""
    np.random.seed(seed)
    idx = np.random.permutation(len(y))
    cut = int(ratio * len(y))
    return tx[idx[:cut]], y[idx[:cut]], tx[idx[cut:]], y[idx[cut:]]

tx_tr, y_tr, tx_val, y_val = split_data(y, tx_train, ratio=0.8)

TRAIN A MODEL

In [15]:
initial_w = np.zeros(tx_tr.shape[1])
max_iters = 200
gamma = 0.1
lambda_ = 1e-4

w, loss = reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)
print("train loss:", loss)

train loss: 0.2426247628037984


EVALUATE ON VALIDATION SET

In [16]:
def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

def predict_labels(w, tx, threshold=0.5):
    """Return predictions in {-1, 1} (format required by the submission)."""
    return np.where(sigmoid(tx @ w) >= threshold, 1, -1)

def f1_score(y_true, y_pred):
    """F1 score for labels in {-1, 1}."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return 2 * tp / (2 * tp + fp + fn + 1e-12)

y_val_pm = 2 * y_val - 1             # back to {-1, 1}
y_pred_val = predict_labels(w, tx_val)
print("accuracy:", np.mean(y_pred_val == y_val_pm))
print("F1 score:", f1_score(y_val_pm, y_pred_val))

accuracy: 0.9157511390129063
F1 score: 0.13161614575153133


CREATE THE SUBMISSION

In [18]:
y_pred = predict_labels(w, tx_test)
create_csv_submission(test_ids, y_pred, "submission1.csv")

REMOVE CONSTANT COLUMN

In [19]:
# After standardization, constant columns are all zeros, so tx^T tx is singular
keep = tx_tr.std(axis=0) > 0
keep[0] = True                      # always keep the offset column
tx_tr, tx_val, tx_test = tx_tr[:, keep], tx_val[:, keep], tx_test[:, keep]
print("features kept:", tx_tr.shape[1])

features kept: 316


TRAIN 6 MODELS

In [20]:
initial_w = np.zeros(tx_tr.shape[1])

# 1-2. Linear regression, iterative (small gamma: MSE is not normalized like logistic)
w_gd, _  = mean_squared_error_gd(y_tr, tx_tr, initial_w, max_iters=200, gamma=0.01)
w_sgd, _ = mean_squared_error_sgd(y_tr, tx_tr, initial_w, max_iters=50000, gamma=0.001)

# 3-4. Linear regression, normal equations
w_ls, _    = least_squares(y_tr, tx_tr)
w_ridge, _ = ridge_regression(y_tr, tx_tr, lambda_=1e-3)

# 5-6. Logistic regression
w_log, _    = logistic_regression(y_tr, tx_tr, initial_w, max_iters=200, gamma=0.1)
w_reglog, _ = reg_logistic_regression(y_tr, tx_tr, 1e-4, initial_w, max_iters=200, gamma=0.1)

EVALUATE AND PRINT TABLE 

In [21]:
def predict_linear(w, tx, threshold=0.5):
    """Linear models output a real number: compare it to the threshold."""
    return np.where(tx @ w >= threshold, 1, -1)

def best_threshold(w, predict_fn, grid=np.linspace(0.05, 0.95, 19)):
    """Threshold on the validation set that maximizes F1."""
    scores = [f1_score(y_val_pm, predict_fn(w, tx_val, t)) for t in grid]
    return grid[int(np.argmax(scores))]

models = {
    "mean_squared_error_gd":   (w_gd,     predict_linear),
    "mean_squared_error_sgd":  (w_sgd,    predict_linear),
    "least_squares":           (w_ls,     predict_linear),
    "ridge_regression":        (w_ridge,  predict_linear),
    "logistic_regression":     (w_log,    predict_labels),
    "reg_logistic_regression": (w_reglog, predict_labels),
}

print(f"{'model':26s} {'acc@0.5':>8s} {'F1@0.5':>8s} {'best thr':>9s} {'F1@best':>8s}")

# Baseline: always predict "no heart attack"
y_base = -np.ones_like(y_val_pm)
print(f"{'always -1 (baseline)':26s} {np.mean(y_base == y_val_pm):8.3f} "
      f"{f1_score(y_val_pm, y_base):8.3f}")

for name, (w_m, predict_fn) in models.items():
    y_pred = predict_fn(w_m, tx_val)                 # default threshold 0.5
    t = best_threshold(w_m, predict_fn)
    y_pred_t = predict_fn(w_m, tx_val, t)
    print(f"{name:26s} {np.mean(y_pred == y_val_pm):8.3f} {f1_score(y_val_pm, y_pred):8.3f} "
          f"{t:9.2f} {f1_score(y_val_pm, y_pred_t):8.3f}")

model                       acc@0.5   F1@0.5  best thr  F1@best
always -1 (baseline)          0.914    0.000
mean_squared_error_gd         0.914    0.008      0.20    0.400
mean_squared_error_sgd        0.505    0.159      0.05    0.159
least_squares                 0.914    0.022      0.20    0.402
ridge_regression              0.914    0.017      0.20    0.400
logistic_regression           0.916    0.132      0.25    0.402
reg_logistic_regression       0.916    0.132      0.25    0.403
